# Analyse approfondie des retraits terminal et détection des anomalies

**Objectif.** Comprendre le comportement normal de chaque terminal et faire émerger toutes les anomalies, y
compris les plus cachées, au grain de la transaction. Trois familles de détection :
- **anomalies directes (01 à 11)** : un indicateur sort nettement de l'habitude du terminal ;
- **anomalies cachées (12 à 17)** : rien n'est extrême pris séparément, mais la forme de la journée, le
  mélange des réseaux, l'écart aux terminaux comparables, une dérive lente, un trou horaire récurrent ou une
  combinaison de petits écarts trahit le problème ;
- **anomalies avancées (18 à 26)** : cassette vide, incident de zone, attaque coordonnée, motifs de capture,
  échecs d'un réseau, rythme trop régulier, heure morte, même carte sur plusieurs terminaux, écart à l'année N-1.

Le catalogue repère beaucoup de journées atypiques ; la **priorisation** (section 11) ne transmet au métier
que les plus graves, dans la limite de sa capacité de traitement quotidienne.

**Organisation.** Tout le code est dans la librairie `analyse_terminaux` (bibliothèque Python du projet Dataiku) :
configuration, préparation, indicateurs, règles d'anomalies, graphiques. Ce notebook ne contient que la
configuration, les appels et la lecture des résultats. Pour l'adapter à une autre table : section 1.

**Toutes les lignes sont utilisées.** Les calculs sont faits par Spark sur la table entière ; seuls des
résultats agrégés sont ramenés en pandas. Kernel : environnement **distrib** (ex. `python36distrib_new`).

## 0. Imports et session Spark

In [ ]:
import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from analyse_terminaux import analyses_metier, anomalies, priorisation
from analyse_terminaux import graphiques as gr
from analyse_terminaux import indicateurs as ind
from analyse_terminaux import preparation as prep
from analyse_terminaux.configuration import configuration
from analyse_terminaux.outils import CLE_TERMINAL, en_pandas, fmt_nombre

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 1. Configuration
**Toutes les valeurs ci-dessous sont FICTIVES** (format de la table de test). Remplacer chaque valeur entre
guillemets par celle de la vraie table : nom du dataset, noms des colonnes, codes réseau, libellés.
Seule la partie droite change ; les clés à gauche sont utilisées par le code et ne se modifient pas.
Les seuils des règles se règlent dans `seuils={...}` (liste complète : `analyse_terminaux/configuration.py`).

In [ ]:
DATASET_OPERATIONS = "retraits_terminal_fictif"   # nom du dataset dans le Flow Dataiku
TABLE_HIVE_OPERATIONS = None                 # ou nom Hive complet "base.table" (si pas de dataset)
DATASET_SORTIE_TRANSACTIONS = None           # ex. "retraits_enrichis" : enregistre la table enrichie (section 4)

# --- noms des colonnes : code -> nom dans la table
COLONNES = {
    "num_automate": "id_terminal",                       # identifiant du terminal
    "date_operation": "date_ope",                   # date de l'opération (format : FORMAT_DATE)
    "heure_operation": "heure_ope",                 # heure HHMMSS (les zéros de tête peuvent manquer)
    "montant_ope": "montant_centimes",              # montant (en centimes : DIVISEUR_MONTANT = 100)
    "lib_ope": "libelle_ope",                       # libellé de l'opération (retrait, capture...)
    "type_reseau": "code_reseau",                   # code réseau de la carte
    "code_carte_privative": "code_carte_speciale",  # code de carte privative / spéciale
    "top_annule": "flag_annulation",                # 1 = opération annulée
    "lib_motif_capture_carte": "motif_capture",     # motif de capture de la carte
    "date_validite_carte": "fin_validite_carte",    # fin de validité MMAA
    "code_site": "code_site",                       # code du type de site d'implantation
    "libelle_site": "type_site",                    # libellé du type de site (agence, gare...)
    "code_postal": "cp",
    "ville": "commune",
    "longitude": "lon",
    "latitude": "lat",
    "nb1": "nb_billets_1", "nb2": "nb_billets_2", "nb3": "nb_billets_3", "nb4": "nb_billets_4",
    "coupure1": "valeur_billet_1", "coupure2": "valeur_billet_2",
    "coupure3": "valeur_billet_3", "coupure4": "valeur_billet_4",
    "id_carte": "id_carte",                         # facultatif (règle 25) : laisser si la colonne n'existe pas
}

# --- règles de gestion : codes réseau et libellés
CODES_RESEAU = {        # code réseau -> nom du réseau (codes sur 2 caractères : "1" s'écrit "01")
    "10": "Reseau national",
    "20": "Reseau international 1",
    "30": "Reseau international 2",
    "40": "Reseau interne",
    "50": "Reseau etranger A",
    "60": "Reseau etranger B",
}
CODES_CARTE_SPECIALE = {   # code carte privative -> nom (utilisé si le code réseau n'est pas dans la liste)
    "SPA": "Reseau etranger C",
    "SPB": "Reseau etranger D",
    "SPC": "Carte privative X",
}
GROUPES_RESEAU = {      # nom du réseau -> groupe affiché dans les graphiques
    "Reseau national": "National",
    "Carte interbancaire": "Interbancaire",
    "Reseau international 1": "International 1",
    "Reseau international 2": "International 2",
    "Reseau etranger A": "Etrangeres rares",
    "Reseau etranger B": "Etrangeres rares",
    "Reseau etranger C": "Etrangeres rares",
    "Reseau etranger D": "Etrangeres rares",
    "Reseau interne": "interne",
}
RESEAU_INTERNE = "Reseau interne"           # sert à séparer captures interne / hors réseau interne
RESEAUX_ETRANGERS_RARES = ["Reseau etranger A", "Reseau etranger B", "Reseau etranger C", "Reseau etranger D"]
TYPES_OPERATION = [     # (motif SQL LIKE sur le libellé en majuscules, type) ; le premier trouvé l'emporte
    ("%AUTRE BANQUE%", "RETRAIT_EXTERNE"),
    ("%CAPTURE%", "CAPTURE"),
    ("%RETRAIT%", "RETRAIT_INTERNE"),
]
MOTIFS_CAPTURE = {      # motif normalisé -> libellés de la table (en majuscules)
    "CARTE_VOLEE": ["MOTIF CARTE VOLEE"],
    "CARTE_PERDUE": ["MOTIF CARTE PERDUE"],
    "CODE_CONFIDENTIEL_DEPASSE": ["MOTIF CODE FAUX"],
    "OUBLI_OU_INCIDENT_LECTURE": ["MOTIF INCIDENT LECTURE"],
    "CONSERVER_LA_CARTE": ["MOTIF CONSERVATION"],
}

# --- formats et période
FORMAT_DATE = "ddMMyyyy"     # format de la colonne date quand elle est en texte
DIVISEUR_MONTANT = 100       # 100 si les montants sont en centimes, 1 s'ils sont en euros
DATE_DEBUT, DATE_FIN = None, None   # ex. "2026-09-01", "2026-09-30" pour un premier essai ; None = toute la table

conf = configuration(
    colonnes=COLONNES,
    metier={"codes_reseau": CODES_RESEAU, "codes_privative": CODES_CARTE_SPECIALE,
            "groupes_reseau": GROUPES_RESEAU, "reseau_interne": RESEAU_INTERNE,
            "reseaux_etrangers": RESEAUX_ETRANGERS_RARES, "types_operation": TYPES_OPERATION,
            "motifs_capture": MOTIFS_CAPTURE},
    options={"format_date": FORMAT_DATE, "diviseur_montant": DIVISEUR_MONTANT,
             "date_debut": DATE_DEBUT, "date_fin": DATE_FIN},
    seuils={},               # ex. {"rafale_min_10min": 15}
)
SEUILS = conf["seuils"]

## 2. Lecture et correspondance des colonnes
Le tableau indique, pour chaque colonne attendue, la colonne retenue dans la table. Une colonne mal trouvée
se corrige dans la configuration (`colonnes=`).

In [ ]:
source = prep.lire_table(DATASET_OPERATIONS, TABLE_HIVE_OPERATIONS)
brut, correspondance = prep.standardiser(source, conf)
display(pd.DataFrame([{"colonne attendue": cle, "colonne retenue": nom or "ABSENTE"}
                      for cle, nom in correspondance.items()]))
retenues = set(n for n in correspondance.values() if n)
print("Colonnes de la table non utilisées :", [c for c in source.columns if c not in retenues] or "aucune")
brut.show(5, truncate=False)
nb_lignes_brut = brut.count()
print("Lignes :", fmt_nombre(nb_lignes_brut))

## 3. Qualité de la donnée brute
Valeurs manquantes, valeurs prises par les colonnes catégorielles, formats, doublons exacts.

In [ ]:
display(ind.qualite_colonnes(brut, nb_lignes_brut))
for colonne in ["lib_ope", "type_reseau", "code_carte_privative", "top_annule", "lib_motif_capture_carte",
                "coupure1", "coupure2", "coupure3", "coupure4", "libelle_site"]:
    if correspondance.get(colonne):
        print("\n=== {} ===".format(colonne))
        display(ind.valeurs_prises(brut, colonne, nb_lignes_brut))

In [ ]:
date_op = prep.expression_date(dict(brut.dtypes)["date_operation"], conf["options"]["format_date"])
formats = ind.controle_formats(brut, date_op, prep.expression_heure())
for cle, valeur in formats.items():
    print("{:<28}: {}".format(cle, valeur))
print("Montant médian : {:.2f} €".format((formats["montant_median_brut"] or 0) / conf["options"]["diviseur_montant"]))
nb_doublons = nb_lignes_brut - brut.distinct().count()
print("Lignes en double exact : {} ({:.3f} %)".format(fmt_nombre(nb_doublons), 100 * nb_doublons / nb_lignes_brut))

## 4. Préparation : une ligne par opération, enrichie
Chaque opération reçoit ses propres indicateurs (type, réseau, inabouti, capture interne / hors réseau interne) puis des
indicateurs calculés à partir de ses voisines sur le même terminal : délai depuis l'opération précédente,
opérations à +/- 5 minutes, même montant que la précédente, captures dans l'heure, rang dans la journée.

In [ ]:
ops = prep.construire_operations(brut, correspondance, conf)
resume = prep.resumer(ops)
display(en_pandas(brut.groupBy("lib_ope", prep.expression_type_operation(conf["metier"]).alias("type_op"))
                  .count().orderBy(F.desc("count"))))
transactions = prep.enrichir_transactions(ops, conf["options"]["persister"])
journees = prep.vue_terminal_jour(ops, conf["options"]["persister"])
referentiel = prep.referentiel_terminal(brut, correspondance)
retraits = ops.filter(F.col("est_retrait") == 1)
validite = prep.lire_validite(retraits)
donnees = anomalies.DonneesAnalyse(ops, transactions, journees, conf, validite, referentiel, correspondance)
exemple = transactions.select(CLE_TERMINAL).limit(1).collect()[0][0]
display(en_pandas(transactions.filter(F.col(CLE_TERMINAL) == exemple).orderBy("t").limit(20)
                  .select(CLE_TERMINAL, "ts", "type_op", "cat_reseau", "montant", "montant_billets", "inabouti",
                          "delai_precedente_s", "nb_ope_5min", "capture_dans_l_heure")))
if DATASET_SORTIE_TRANSACTIONS:
    import dataiku
    import dataiku.spark as dkuspark
    dkuspark.write_with_schema(dataiku.Dataset(DATASET_SORTIE_TRANSACTIONS), transactions.drop("t"))

### Indicateurs par réseau, comptés transaction par transaction

In [ ]:
display(ind.indicateurs_par_reseau(ops))

## 5. Volumes dans le temps
**Comment lire :** une chute isolée du parc entier signale un événement commun (incident réseau, jour
férié) ; elle ne doit pas être imputée aux terminaux individuellement.

In [ ]:
jour = ind.volumes_par_jour(ops)
gr.tracer_volumes(jour)
print("Journées du parc les plus éloignées de la normale du même jour de semaine :")
display(jour.reindex(jour["ratio_vs_meme_jour"].sub(1).abs().sort_values(ascending=False).index).head(15)
        [["jour_semaine", "ferie", "retraits", "ratio_vs_meme_jour", "terminal_actifs", "captures", "inaboutis"]])
gr.tracer_calendrier(jour["retraits"], "Calendrier du parc : retraits par jour")
gr.tracer_matrice(ind.croisement_jour_heure(retraits, jour.groupby("jour_semaine").size()),
                  "Retraits moyens par jour : jour de semaine x heure", xlabel="heure")

## 6. Comportements types des terminaux
Profils normalisés (part des retraits par heure et par jour de semaine), une ligne par terminal : les terminaux qui se
ressemblent forment des bandes ; un terminal qui ne ressemble à aucun autre ressort.

**Carte du parc** : une ligne par terminal, une colonne par jour, couleur = retraits / habitude du terminal pour ce jour
de semaine (log2). Trait vertical = événement commun ; trait horizontal = terminal durablement anormal ; point
isolé = anomalie ponctuelle.

In [ ]:
profil_horaire = ind.profils_normalises(retraits, "heure", range(24))
profil_horaire = profil_horaire.iloc[pd.Series(profil_horaire.values.argmax(axis=1)).argsort().values]
profil_hebdo = ind.profils_normalises(retraits, "jour_semaine", ind.ORDRE_JOURS)
profil_hebdo = profil_hebdo.iloc[(profil_hebdo["sam"] + profil_hebdo["dim"]).argsort().values]
gr.tracer_profils(profil_horaire, profil_hebdo)
journees_pd = en_pandas(journees, max_lignes=20000000)
journees_pd["date"] = pd.to_datetime(journees_pd["date"])
gr.tracer_carte_parc(ind.carte_parc(journees_pd))

## 7. Le parc, sa géographie et les relations entre indicateurs

In [ ]:
par_terminal = ind.statistiques_par_terminal(ops).merge(en_pandas(referentiel), on=CLE_TERMINAL, how="left")
display(par_terminal.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.round(2))
gr.tracer_parc(par_terminal)
geo = prep.geographie_presente(correspondance)
if geo:
    display(ind.controles_geographiques(par_terminal, conf["options"]["metropole"]))
    gr.tracer_cartes_geographiques(ind.filtrer_metropole(par_terminal, conf["options"]["metropole"]))
if "libelle_site" in par_terminal.columns:
    gr.tracer_volume_par_site(par_terminal)
    gr.tracer_relations(par_terminal[par_terminal["retraits"] >= 500], "libelle_site")

## 8. Montants, réseaux, inaboutis, captures, validité des cartes

In [ ]:
gr.tracer_montants(ind.tranches_montants(retraits), ind.quantiles_montant(retraits, "heure"),
                   ind.croisement_jour_heure(retraits, None, valeur="montant"))
display(ind.montants_frequents(retraits, resume["retraits"]))
print(ind.controles_montants(retraits))
display(ind.ecarts_billets(retraits))

In [ ]:
gr.tracer_reseaux(ind.repartition_reseaux(retraits, "mois"), ind.profils_reseaux(retraits, "heure"))
gr.tracer_histogramme(par_terminal.loc[par_terminal["retraits"] >= 1000, "part_etrangers_pct"],
                      "Part de cartes étrangères rares par terminal (%)")

In [ ]:
gr.tracer_taux(ind.taux_par(retraits, "heure", "inabouti"), ind.taux_par(retraits, "groupe_reseau", "inabouti"),
               "Taux d'inaboutis")
display(ind.taux_captures(ops))
gr.tracer_histogramme(par_terminal.loc[par_terminal["retraits"] >= 500, "taux_inaboutis_pct"], "Taux d'inaboutis par terminal (%)")
if validite is not None:
    expirees = validite.agg(F.avg((F.col("mois_restants") < 0).cast("int"))).collect()[0][0]
    print("Retraits avec une carte expirée : {:.3f} %".format(100 * (expirees or 0)))

## 9. Dynamique à l'intérieur de la journée
**Silences** : heures consécutives sans retrait, comparées au rythme horaire habituel du terminal recalé sur
l'activité du jour ; probabilité qu'un tel silence arrive par hasard = exp(-attendu) (loi de Poisson).

In [ ]:
retraits_tx = donnees.retraits
display(ind.tranches_delais(retraits_tx))
silences = donnees.silences()
improbables = silences.filter(F.col("proba_hasard") < SEUILS["silence_proba"])
print("Journées terminal avec un silence improbable :", fmt_nombre(improbables.count()))
gr.tracer_silences_et_delais(ind.silences_debut_duree(improbables), ind.delais_echelle_log(retraits_tx))
gr.tracer_ecart_parc_heure(ind.ecart_parc_jour_heure(retraits))

## 10. Catalogue des anomalies

| # | Anomalie | Ce qui la déclenche |
|---|---|---|
| 01 | Silence anormal | heures sans retrait, probabilité < 1/100 000 vu le rythme du terminal |
| 02 | Journée sans opération | aucun retrait, seulement pour un terminal très fréquenté (top 10 %) |
| 03 | Rafale | >= 10 opérations en 10 min et x2 le niveau élevé habituel du terminal |
| 04 | Même montant répété | >= 6 retraits du même montant en 5 minutes |
| 05 | Inaboutis en série | >= 3 retraits inaboutis consécutifs |
| 06 | Captures hors réseau interne en série | >= 3 captures hors réseau interne en une heure |
| 07 | Afflux de cartes étrangères | réseaux étrangers rares (configuration) : >= 20 en 1 h et x5 l'habitude |
| 08 | Nuit inhabituelle | >= 3 retraits entre 0 h et 5 h sur un terminal qui n'en fait presque jamais |
| 09 | Carte expirée | date de validité dépassée |
| 10 | Volume journalier anormal | retraits x3 vs 8 mêmes jours de semaine et 7 derniers jours |
| 11 | Montant moyen anormal | montant moyen x3 vs 8 mêmes jours de semaine |
| 12 | Forme horaire inhabituelle | répartition des retraits dans la journée inhabituelle (Jensen-Shannon) |
| 13 | Mélange de réseaux inhabituel | part des réseaux de cartes inhabituelle (Jensen-Shannon) |
| 14 | Écart aux terminaux pairs | le terminal décroche (< 40 %) quand ses pairs restent normaux, ou l'inverse |
| 15 | Dérive lente | 4 semaines < 60 % des 8 précédentes, sans arrêt net, pairs stables |
| 16 | Trou horaire récurrent | une heure active vide plus d'un jour sur deux dans le mois |
| 17 | Combinaison de petits écarts | Isolation Forest sur 9 écarts robustes, journées non vues ailleurs |
| 18 | Cassette vide ou défaillante | une coupure habituelle (>= 10 % des billets) disparaît des billets délivrés |
| 19 | Incident de zone | >= 5 terminaux d'un même département en silence improbable à la même heure |
| 20 | Attaque coordonnée | rafale, montant répété ou cartes étrangères sur >= 3 terminaux d'une zone dans l'heure |
| 21 | Motif de capture inhabituel | >= 3 captures d'un motif, probabilité de hasard < 1/10 000 (Poisson) |
| 22 | Inaboutis concentrés sur un réseau | >= 30 % d'échecs sur un réseau, les autres réseaux < 10 % |
| 23 | Rythme trop régulier | même délai (à la seconde) entre deux retraits >= 8 fois en 30 min, hasard < 1e-6 |
| 24 | Activité à une heure morte | >= 3 retraits à une heure active moins de 1 % des jours |
| 25 | Même carte sur plusieurs terminaux | vitesse de déplacement > 300 km/h (identifiant de carte nécessaire) |
| 26 | Écart à l'année précédente | retraits x3 ou /3 vs l'année précédente (plus d'un an de données) |

Seuils : `configuration.py` (surcharges dans la section 1). Pour ajouter une anomalie : une fonction
décorée par `@regle("NN", "Nom")` dans `anomalies.py` ou `anomalies_cachees.py`.

In [ ]:
evenements = anomalies.detecter_anomalies(donnees, SEUILS)
print("Événements repérés :", fmt_nombre(evenements.count()))
recap, par_type_terminal = anomalies.synthese_par_type(evenements, len(par_terminal), resume["jours"])
display(recap.round(2))
gr.tracer_frequences(recap)
gr.tracer_recurrence(par_type_terminal)

### 10.1 Intensité, évolution et cumul des anomalies
**Cumul** : part des journées de l'anomalie (ligne) qui présentent aussi l'anomalie (colonne). Une forte
valeur indique deux symptômes d'une même cause.

In [ ]:
evenements_pd = en_pandas(evenements.select(CLE_TERMINAL, "date", "type", "valeur"), max_lignes=20000000)
evenements_pd["date"] = pd.to_datetime(evenements_pd["date"])
gr.tracer_intensites(evenements_pd)
gr.tracer_evolution(evenements_pd.groupby(["date", "type"]).size().rename("count").reset_index())
cumul = anomalies.cooccurrences(evenements)
gr.tracer_matrice(cumul, "% des journées de l'anomalie (ligne) avec aussi l'anomalie (colonne)", cmap="Purples",
                  taille=(12, 9), annoter=True, etiquettes_x=[t[:2] for t in cumul.columns], vmax=60)

### 10.2 Anomalies cachées : ce que les règles directes ne voient pas
Journées signalées **uniquement** par les règles 12 à 17. Pour la règle 17, les indicateurs en cause sont
donnés en « écarts habituels » (+ = au-dessus de l'habitude du terminal).

In [ ]:
codes_directs = ["{:02d}".format(i) for i in range(1, 12)]
journees_directes = set(map(tuple, evenements_pd[evenements_pd["type"].str[:2].isin(codes_directs)]
                            [[CLE_TERMINAL, "date"]].values))
cachees = evenements_pd[~evenements_pd["type"].str[:2].isin(codes_directs)].copy()
cachees["vue_aussi_par_une_regle_directe"] = [(g, d) in journees_directes
                                              for g, d in zip(cachees[CLE_TERMINAL], cachees["date"])]
display(cachees.groupby("type")["vue_aussi_par_une_regle_directe"].agg(["size", "mean"])
        .rename(columns={"size": "journees", "mean": "part_vue_par_regle_directe"}).round(2))
scores = donnees.resultat("scores_multivaries")
if scores is not None:
    gr.tracer_explications(scores, SEUILS["multivarie_part"])
    display(scores.nlargest(15, "score")[[CLE_TERMINAL, "date", "score", "explication"]])

### 10.3 Où sont les anomalies ?

In [ ]:
synthese_terminal, occurrences = anomalies.synthese_par_terminal(par_type_terminal, par_terminal)
if geo:
    positions = ind.filtrer_metropole(par_terminal.set_index(CLE_TERMINAL)[["longitude", "latitude"]],
                                      conf["options"]["metropole"])
    gr.tracer_cartes_anomalies(occurrences, positions)
if "libelle_site" in par_terminal.columns:
    site = par_terminal.set_index(CLE_TERMINAL)["libelle_site"]
    part_site = (occurrences > 0).join(site).groupby("libelle_site").mean().reindex(
        site.value_counts().head(12).index) * 100
    gr.tracer_matrice(part_site, "% des terminaux de chaque type de site touchés par chaque anomalie", cmap="Oranges",
                      taille=(16, 6), annoter=True, etiquettes_x=[t[:2] for t in part_site.columns])

### 10.4 Les terminaux les plus atypiques : galerie, chronologie, détail

In [ ]:
display(synthese_terminal.head(30)[[c for c in ("ville", "libelle_site", "retraits_par_jour_actif")
                               if c in synthese_terminal.columns] + ["nb_anomalies_differentes", "evenements", "detail"]])
gr.tracer_galerie(journees_pd, evenements_pd, list(synthese_terminal.index[:12]), synthese_terminal)
gr.tracer_chronologie(evenements_pd, list(synthese_terminal.sort_values("evenements", ascending=False).index[:40]))

### 10.5 Un exemple réel par anomalie
Le cas le plus marqué de chaque anomalie : la journée du terminal, quart d'heure par quart d'heure, comparée aux
8 mêmes jours de semaine précédents. Pour approfondir un terminal : `gr.carte_heures_terminal(ops, "terminal...")`.

In [ ]:
exemples = anomalies.exemples_par_type(evenements)
display(exemples)
for _, cas in exemples.iterrows():
    print("\n=== {} - terminal {} le {} (valeur {:.2f}) ===".format(cas["type"], cas[CLE_TERMINAL], cas["date"], cas["valeur"]))
    gr.regarder_journee(ops, cas[CLE_TERMINAL], str(cas["date"]))

### 10.6 Choisir les anomalies à retenir
Une anomalie utile est assez fréquente, touche plusieurs terminaux et revient, sans être si fréquente qu'elle
décrive en fait un comportement normal.

In [ ]:
display(anomalies.aide_au_choix(recap).round(2))

## 11. Priorisation : ce qui est réellement transmis au métier
Le catalogue repère toutes les journées atypiques ; les transmettre toutes noierait le métier. Pour le jour
analysé (J-3), chaque terminal alerté reçoit un **score de risque** :

score = somme des gravités des anomalies du jour (1 à surveiller, 2 à traiter, 3 urgent)
        x (1 + 0,1 x nombre de jours alertés sur les 28 derniers jours) x 0,5 si jour férié ou particulier.

Les symptômes d'un terminal pris dans un incident de zone sont regroupés dans une seule alerte de zone. Niveaux :
Modéré (< 3), Élevé (3 à 6), Critique (>= 6). Seuls les niveaux Élevé et Critique sont transmis, dans la
limite de `BUDGET_ALERTES_JOUR` (capacité de traitement du métier).

In [ ]:
BUDGET_ALERTES_JOUR = 30
DATE_ANALYSE = None          # None = dernier jour de la table ; en production : J-3
alertes = priorisation.score_de_risque(evenements_pd, DATE_ANALYSE,
                                       autres_jours_particuliers=conf["options"]["jours_particuliers"])
transmises = priorisation.selection_quotidienne(alertes, BUDGET_ALERTES_JOUR)
print("terminal analysés : {} | terminal avec au moins une anomalie ce jour : {} | alertes transmises : {}".format(
    len(par_terminal), len(alertes), len(transmises)))
print(alertes["niveau"].value_counts().to_string())
tableau = priorisation.tableau_metier(transmises, par_terminal)
display(tableau)

### Combien d'alertes chaque jour ?
Nombre de terminaux alertés par jour et par niveau sur toute la période : c'est la charge que représenterait
chaque niveau pour le métier. Le budget quotidien plafonne ce qui est transmis.

In [ ]:
charge = pd.concat([priorisation.score_de_risque(evenements_pd, jour_analyse)
                    .assign(jour=jour_analyse)[["jour", "niveau"]]
                    for jour_analyse in sorted(evenements_pd["date"].unique())[-60:]])
gr.tracer_charge_quotidienne(charge, BUDGET_ALERTES_JOUR)

### Mesurer la qualité avec le métier
Le tableau transmis contient une colonne `avis_metier` (« vraie anomalie » / « fausse alerte »). Une fois
remplie et rechargée, `priorisation.precision_par_regle(evenements_pd, retours)` donne la précision de chaque
règle, pour ajuster ses seuils ou sa gravité.

## 12. Ce que l'analyse apprend au métier
Au-delà des alertes, les mêmes transactions répondent à des questions d'exploitation.

### 12.1 Saturation : quels terminal atteignent leur capacité ?
Capacité retenue : 30 retraits par heure (environ 2 minutes par client). Un terminal souvent à capacité fait
attendre ses clients : candidat à un second terminal.

In [ ]:
saturation = analyses_metier.saturation_horaire(retraits)
display(saturation.head(20))
gr.tracer_histogramme(saturation["heure_chargee_p99"], "Retraits de l'heure la plus chargée (99e centile), par terminal")

### 12.2 Besoin en billets par coupure et par jour de semaine
Billets délivrés en moyenne par cassette : base pour planifier les chargements (jour et coupures à prévoir).

In [ ]:
billets = analyses_metier.besoin_billets(ops)
billets["jour"] = billets["jour_semaine"].map(ind.JOURS_FR)
parc_billets = billets.groupby("jour")[[c for c in billets.columns if c.startswith("billets_nb")]].sum()
gr.tracer_matrice(parc_billets.reindex(ind.ORDRE_JOURS).T, "Billets délivrés par jour, parc complet : cassette x jour",
                  cmap="Greens", taille=(12, 3.5), annoter=True)
display(billets.sort_values("montant_moyen_delivre", ascending=False).head(15))

### 12.3 Créneau de maintenance : l'heure la plus creuse de chaque terminal (6 h - 22 h)

In [ ]:
creneaux = analyses_metier.creneau_maintenance(retraits)
display(creneaux["heure_la_plus_creuse"].value_counts().sort_index().rename("terminal").to_frame().T)
display(creneaux.head(15))

### 12.4 Fiabilité, tendance et temps entre pannes
Incidents (pannes, inaboutis, captures, cassettes) pour 1 000 retraits ; tendance > 1 : le terminal se dégrade
depuis 8 semaines (maintenance préventive à prévoir).

In [ ]:
fiabilite = analyses_metier.fiabilite(evenements_pd, par_terminal)
print("terminal qui se dégradent (tendance >= 2) :", int((fiabilite["tendance"] >= 2).sum()))
display(fiabilite.head(20))
display(analyses_metier.temps_entre_pannes(evenements_pd).head(20))

### 12.5 Usage par rapport aux terminaux comparables
Retraits par jour du terminal rapportés à la médiane de son type de site : sous-utilisé (< 0,3) ou très
sollicité (> 3).

In [ ]:
display(analyses_metier.usage_par_rapport_aux_pairs(par_terminal))

In [ ]:
for table in (evenements, journees, transactions, ops):
    table.unpersist()